In [ ]:
import flow
import numpy as np
import pandas as pd
import os
import sys
import subprocess
import shutil
import matplotlib.pyplot as plt
from flow import FlowProject, directives
from pymser import pymser
import warnings
import panedr
from block_average.block_average import block_average

In [ ]:
from utils.molec_class_files import esolvs
dict = esolvs.make_dict()
for i, j in dict.items():
    rho_avg = (min(list(j.expt_liq_density.values()))+ max(list(j.expt_vap_density.values())))/2
    density = max(list(j.expt_liq_density.values()))
    #Calculate the box lengths from the system density
    xy_len = 3.6 #*np.max(j.bounds_sig)
    new_V = 3*xy_len**3
    #Calculate the number of molecules from the new volume and the given density
    nmols = int(np.floor(density*1000*6.022*1e23*new_V/(j.molecular_weight*1e27)))
    V2 = (nmols*j.molecular_weight*1e27)/(rho_avg* 1000* 6.022*1e23)

    # print(i, V1**(1/3), V2/(V1**(2/3)), (V2/(V1**(2/3)))/V1**(1/3))
    # print(i, boxy, boxz, box3, nmols, nmols1, nparts)
    z = 3*xy_len
    z2 = V2/(xy_len**2)
    print(i, xy_len, z, z2, nmols)

In [ ]:
from utils.molec_class_files import esolvs
dict = esolvs.make_dict()
for i, j in dict.items():
    nmols = 20000// j.n_atoms
    nmols = 3000
    nparts = nmols * j.n_atoms

    rho_avg = (min(list(j.expt_liq_density.values()))+ max(list(j.expt_vap_density.values())))/2
    nmols = 2000 #Use no fewer than 2000 molecules (8000 particles)
    density = max(list(j.expt_liq_density.values()))
    #Calculate the box lengths from the system density using 2000 molecules
    V = (nmols*j.molecular_weight*1e27)/(density * 1000* 6.022*1e23)
    V2 = (nmols*j.molecular_weight*1e27)/(density *3* 1000* 6.022*1e23)
    xy_len = (V/3)**(1/3)
    
    #If 2000 molecules is not enough to satisfy xy_len > 13.2*max_sigma
    if xy_len < 13.2*np.max(j.bounds_sig):
        #Calculatue box lengths from system density and 13.2*max_sigma
        xy_len = 13.2*np.max(j.bounds_sig)
        new_V = 3*xy_len**3
        #Calculate the number of molecules from the new volume and the given density
        nmols = int(np.floor(density*1000*6.022*1e23*new_V/(j.molecular_weight*1e27)))

    # print(i, V1**(1/3), V2/(V1**(2/3)), (V2/(V1**(2/3)))/V1**(1/3))
    # print(i, boxy, boxz, box3, nmols, nmols1, nparts)
    z = 3*xy_len
    print(i, xy_len, z, z*(V2/V))

In [ ]:
file = "example_output_files/sft_calc.edr"
edr = panedr.edr_to_df(file)

In [ ]:
edr.tail()

In [ ]:
dt = edr["Time"].values
print(dt)
nsteps = 70000000
tot_rows = 70000
prod_step = 10000
nprod = tot_rows*prod_step
print(nprod)
print(nsteps/tot_rows)
print(nsteps/prod_step)

In [ ]:
file = np.loadtxt("density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_prod_density.xvg", comments=["#", "@"])
df = pd.DataFrame(file, columns=["Time", "RMSD"])
z = data[0]  # z-position
density = data[1]  # raw density

# Apply running average with window size = 5 bins
smoothed_density = density.rolling(window=5, center=True).mean()

# Plot
plt.plot(z, density, label="Raw")
plt.plot(z, smoothed_density, label="Running Avg", linewidth=2)
plt.legend()
plt.xlabel("z (nm)")
plt.ylabel("Density (kg/m^3)")
plt.show()

In [ ]:
file = np.loadtxt("density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_prod_rmsd.xvg", comments=["#", "@"])
df = pd.DataFrame(file, columns=["Time", "RMSD"])
plt.plot(df["Time"], df["RMSD"])

In [ ]:
# Get the density and volume data
sim_name = "sft_calc"
# file = "example_output_files/" + sim_name + ".edr"
# file = "density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod.edr"
file = "density_iters/runs_npzzat_rect/workspace/e211d0387e1ce02d02bd11448d6088fe/inter_eq.edr"
file = "density_iters/config_copy/workspace/31ba83501abc5d26935784563c3decfa/inter_eq.edr"
file = "density_iters/config_copy/workspace/7b4a706282ef812bf3592491245e812c/inter_eq.edr"
file = "density_iters/runs_npzzat_rect/workspace/301d869508a6f0f95a991fff8935e7f2/npzzat_prod.edr"
# file = "density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_eq.edr"
file = "Build_GPs/vle_iters/workspace/ea11c473c1db53c4e4f945baba2ae18b/calc_props/ift_liq_dens_1.xvg"
# file = "example_output_files/sft_calc.edr"
df_all = panedr.edr_to_df(file)
df = df_all
# file1 = "example_output_files/prd_npt.edr"
# df_all1 = panedr.edr_to_df(file)
# df_all2 = panedr.edr_to_df(file1)
# df_all2["Time"] = df_all2["Time"] + df_all1["Time"].iloc[-1]
# df_all = pd.concat([df_all1, df_all2], ignore_index=True)
print(df_all.columns)
eq_data_dict = {}
property = "Density" #"Constr. rmsd"
if property in df_all.columns:
    df = df_all[["Time", property]].copy()
    print(df)

# df = df_all[["Time", "Pres-XX", "Pres-ZZ", "Pres-YY"]].copy()
# df["Pres-N"] = df["Pres-ZZ"] - (df["Pres-XX"] + df["Pres-YY"])/2

# elif property in ["Total Energy", "Volume", "Pressure", "#Surf*SurfTen"]:
    # command = "source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC"
    # subprocess.run(command, shell=True, check=True)
    # print(os.path.exists(os.path.join("example_output_files", f"{sim_name}.edr")))
    # command = (
    #     f"source /afs/crc.nd.edu/group/maginn/group_members/Ryan_DeFever/software/gromacs-2020/gromacs-dp/bin/GMXRC && module load gromacs && gmx_d energy -f example_output_files/{sim_name}.edr -s example_output_files/{sim_name}.tpr -o {sim_name}_{property}.xvg"
    # )
    # subprocess.run(command, input=f"{property}\n", text=True, check=True, shell = True)
    # prop_data = np.loadtxt(
    #     sim_name + "_" + property + ".xvg", comments=["#", "@"]
    # )

    # df = pd.DataFrame(prop_data)

# time_data = df.iloc[:int(len(df)*0.7), 0].values
# property_data = df.iloc[:int(len(df)*0.7), 1].values
time_data = df.iloc[:, 0].values
property_data = df.iloc[:, -1].values
# property_data = df.iloc[:, -1].values #*10**6
print(np.mean(property_data))
plt.plot(time_data, property_data)
print(len(time_data))
eq_col_file = sim_name + "_" + property + ".csv"
eq_data_dict[property] = {
    "data": property_data,
    "time_data": time_data,
    "file": eq_col_file,
}

In [ ]:
def calc_mass_dens(density, mode = "liq"):
    #Find the region attributed to liquid density
    # fraction = 0.4
    # n_shift = int(len(density[:, 1]) * fraction)
    # # Circular shift
    # density[:, 1] = np.roll(density[:, 1], -n_shift)
    
    mass_dens_z = density[:, 1]
    find_liq_slab = find_bulk_liq_index(density, mode = mode)
    range_for_liq_dens = find_liq_slab[0]
    range_org_liq = find_liq_slab[1]
    median_dens_liq = find_liq_slab[3]
    #Calculate the density
    prop_vals = mass_dens_z[range_for_liq_dens] #kg/m^3
    plt.scatter(density[:, 0], density[:, 1], label='ES', color='blue')
    plt.axhline(y=median_dens_liq, color='g', linestyle='--', label= 'Median')
    plt.xlabel('Z (nm)')
    plt.ylabel('Density (kg/m^3)')
    med_dens_pt = find_liq_slab[4]
    
    plt.scatter(density[range_for_liq_dens, 0], density[range_for_liq_dens, 1], label='ES', color='red')
    # plt.scatter(density[range_org_liq, 0], density[range_org_liq, 1], label='ES', color='green')
    plt.scatter(density[med_dens_pt, 0], density[med_dens_pt, 1], label='ES', color='orange')
    plt.show()
    print(np.mean(prop_vals))
    print("Liq Dens Range")
    print(max(density[range_for_liq_dens, 0]), min(density[range_for_liq_dens, 0]))
    return prop_vals
from scipy.signal import find_peaks
from findpeaks import findpeaks


def find_bulk_liq_index(density, mode = "liq"):
    #Use np.diff to approximate the 1st derivative
    ES_numdens_z = density[:,1]
    x = density[:,0]
    # dy = np.diff(ES_numdens_z)
    dy = np.gradient(ES_numdens_z, x)
    # dy = np.gradient(np.gradient(ES_numdens_z, x), x)
    # print(dy)
    # plt.scatter(np.arange(0,len(dy),1), dy, label='ES', color='blue')
    # plt.scatter(results['x'].iloc[interfaces], results['y'].iloc[interfaces], label='ES', color='red')
    #Use findpeaks to find the peaks and valleys, interpolating to get as close as possible
    fp = findpeaks(lookahead=1, interpolate=10, verbose=0)
    results = fp.fit(dy)["df_interp"]

    all_peaks = results[results['peak'] | results['valley']].index.values
    #get the highest peak and the lowest valley, these are the interfaces
    y_vals = results['y'].iloc[all_peaks]
    peak_index = all_peaks[np.argmax(y_vals)]
    valley_index = all_peaks[np.argmin(y_vals)]

    interfaces = [peak_index, valley_index]

    #Divide the indices by 10 to get the correct index for the density based on interpolation
    interfaces = [int(i/10) for i in interfaces]

    #Get the range of indices for the bulk liquid density regardless of whether interfaces have shifted
    if interfaces[0] < interfaces[1]:
        range_org_liq = list(range(interfaces[0], interfaces[1] + 1))
    else:
        range_org_liq = list(range(interfaces[0], len(results)//10)) + list(range(0, interfaces[1] + 1))

    #Get the indecies of the vapor range (the opposite of range_org_liq)
    if interfaces[0] < interfaces[1]:
        range_org_vapor = list(range(interfaces[1] + 1, len(results)//10)) + list(range(0, interfaces[0]))
    else:
        range_org_vapor = list(range(interfaces[1] + 1, interfaces[0]))

    if_r = [peak_index, valley_index]

    # if if_r[0] < if_r[1]:
    #     range_org_liq1 = list(range(if_r[0], if_r[1] + 1))
    # else:
    #     range_org_liq1 = list(range(if_r[0], len(results))) + list(range(0, if_r[1] + 1))

    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[all_peaks]*max(x)/(len(x)*10), results['y'].iloc[all_peaks], label='ES', color='red')
    # plt.scatter(results['x'].iloc[if_r[0]]*max(x)/(len(x)*10), results['y'].iloc[if_r[0]], label='ES', color='green')
    # plt.scatter(results['x'].iloc[if_r[1]]*max(x)/(len(x)*10), results['y'].iloc[if_r[1]], label='ES', color='green')
    
    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # #Divide the indices by 10 to get the correct index for the density based on interpolation
    # interfaces = [int(i/10) for i in if_r]
    # range_org_liq = [int(i/10) for i in range_org_liq1]
    median_dens_liq = np.median(ES_numdens_z[range_org_liq])
    differences = np.abs(ES_numdens_z - median_dens_liq)
    med_dens_pt = np.argmin(differences)
    # plt.scatter(results['x']*max(x)/(len(x)*10), results['y'], label='ES', color='blue')
    # plt.scatter(results['x'].iloc[range_org_liq*10]*max(x)/(len(x)*10), results['y'].iloc[range_org_liq*10], label='ES', color='red')

    median_dens_vap = np.mean(ES_numdens_z[range_org_vapor])
    differences_vap = np.abs(ES_numdens_z - median_dens_vap)
    med_dens_pt_vap = np.argmin(differences_vap)

    # plt.xlabel('Z (nm)')
    # plt.ylabel('d_rho/dz (kg/m^3/nm)')
    # plt.show()

    # Find the first and last index where density >= median
    valid_idx = np.where(ES_numdens_z[range_org_liq] >= median_dens_liq)[0]
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1  # +1 to include the last valid index
        range_for_liq_dens = range_org_liq[start:end]

        # full_range = range_org_liq[start:end]
        # print(len(range_org_liq), len(full_range), len(range_org_liq)/len(full_range))
        # cutoff = int(len(full_range) * 0.85)
        # trim = (len(full_range) - cutoff) // 2
        # range_for_liq_dens = full_range[trim:trim + cutoff]

    else:
        range_for_liq_dens = np.array([], dtype=int)

    median_dens_liq = np.median(ES_numdens_z[range_for_liq_dens])
    differences = np.abs(ES_numdens_z - median_dens_liq)

    #Do the same for vapor
    valid_idx = np.where(ES_numdens_z[range_org_vapor] <= median_dens_vap)[0]
    # print(valid_idx)
    if valid_idx.size > 0:
        start = valid_idx[0]
        end = valid_idx[-1] + 1 # +1 to include the last valid index
        range_for_vapor_dens = range_org_vapor[start:end]
    else:
        range_for_vapor_dens = np.array([], dtype=int)

    median_dens_vap = np.mean(ES_numdens_z[range_for_vapor_dens])
    differences_vap = np.abs(ES_numdens_z - median_dens_vap)
    print("Vap Dens Range")
    idx1_vap = range_org_vapor[0]
    idx2_vap = range_org_vapor[-1]
    print(density[idx1_vap, 0], density[idx2_vap, 0])
    # med_dens_pt = np.argmin(differences)
    # print(range_org_liq)
    # range_for_liq_dens = range_org_liq
    # print(x[max(range_for_liq_dens)], x[min(range_for_liq_dens)])

    if mode == "liq":
        return range_for_liq_dens, range_org_liq, ES_numdens_z, median_dens_liq, med_dens_pt
    elif mode == "vap":
        return range_for_vapor_dens, range_org_vapor, ES_numdens_z, median_dens_vap, med_dens_pt_vap


In [ ]:
#1feec85b78e61abf7d489be38b888430, 0ba3df2e70d9dc80d91472c528393263

# density = np.loadtxt(f"Build_GPs/vle_iters/workspace/78c3344df38e454a54c66a087e8366d6/calc_props/ift_liq_dens.xvg", comments=["#", "@"]) #EG
density = np.loadtxt(f"Build_GPs/vle_iters/workspace/042907986f66368d7fd5e2536a646d49/calc_props/ift_liq_dens.xvg", comments=["#", "@"]) #Gly weird
# density = np.loadtxt(f"Build_GPs/vle_iters/workspace/ecc4080327e453837c8a94b8f0405377/calc_props/ift_liq_dens.xvg", comments=["#", "@"]) #MeOH
# density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_all.xvg", comments=["#", "@"])
prop_vals = calc_mass_dens(density, "liq")
# density = np.loadtxt(file, comments=["#", "@"])
# mean = np.mean(prop_vals)
# print(mean)

In [ ]:
# density = np.loadtxt("density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod_density.xvg", comments=["#", "@"])
# file = "density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg"
# density_org = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
means = []
#5dcc2f1567e58231be94b7091c8ebdd7,1feec85b78e61abf7d489be38b888430, 0ba3df2e70d9dc80d91472c528393263, 1f489249dd7f23f12833e7230f16fa4f, a4ccdf65689e3f5c229bd9960afa9cab
for r in [1,2,3,4,5,6,7,8,9,10]:
    try:
        # density = np.loadtxt(f"density_iters/runs_npzzat/workspace/0ba3df2e70d9dc80d91472c528393263/inter_prod_density_{r}.xvg", comments=["#", "@"])
        density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_{r}.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters/config_copy/workspace/31ba83501abc5d26935784563c3decfa/inter_eq_density_{r}.xvg", comments=["#", "@"])
        prop_vals = calc_mass_dens(density)
        # density = np.loadtxt(file, comments=["#", "@"])
        mean = np.mean(prop_vals)
        means.append(mean)
        print("Run Avg", np.average(means))
    except:
        pass
print("Block avg", np.mean(means))
print(np.var(means)/2)

In [ ]:
# density = np.loadtxt("density_iters/runs/workspace/14d22455341f5c2c9b0ebbc9ba0a905b/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs/workspace/c4e7cea1aaf17ab3c761d6d98f531e6e/inter_prod_density.xvg", comments=["#", "@"])
# file = "density_iters/runs/workspace/0ed4ddd651ded17b529e7e374924bc8e/inter_prod_density.xvg"
# density_org = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
means = []
#5dcc2f1567e58231be94b7091c8ebdd7,1feec85b78e61abf7d489be38b888430, 0ba3df2e70d9dc80d91472c528393263, 1f489249dd7f23f12833e7230f16fa4f, a4ccdf65689e3f5c229bd9960afa9cab
last = None
mapd = []

def relative_abs_error(a,b):
  error = np.abs(a-b)
  rae = error/a
  return rae

for r in range(1,101,1):
    try:
        # density = np.loadtxt(f"density_iters/runs_npzzat_rect/workspace/test/inter_eq_density_{r}_cen.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters/config_copy/workspace/test3/inter_eq_density_{r}_cen.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters/config_copy/workspace/7b4a706282ef812bf3592491245e812c/inter_eq_density_{r}_cen.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters//npt_check_only/workspace/53790f5d62edcf52debeda815e9dc8e4/inter_eq_density_{r}_cen.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_prod_density_{r}.xvg", comments=["#", "@"])
        density = np.loadtxt(f"R125_295_ex/R125_3000_295_rhoMass_profile_System_{r}.xvg", comments=["#", "@"])
        # density = np.loadtxt(f"density_iters/config_copy/workspace/31ba83501abc5d26935784563c3decfa/inter_eq_density_{r}.xvg", comments=["#", "@"])
        df = pd.DataFrame(density, columns=["z", "Density"])
        z = df["z"]  # z-position
        density = df["Density"]  # raw density
        smoothed_density = density.rolling(window=5, center=True).mean()

        if last is not None:
            diff = smoothed_density - last
            # print(np.max(diff))
            means.append(np.max(diff))

            # Plot
            plt.plot(z, density, label="Raw")
            plt.plot(z, smoothed_density, label="Running Avg", linewidth=2)
            plt.plot(z, last, label="Last Running Avg", linewidth=2)
            plt.plot(z, diff, label="Diff Avg", linewidth=2)
            plt.legend()
            plt.xlabel("z (nm)")
            plt.ylabel("Density (kg/m^3)")
            plt.show()
        # else:
        last = smoothed_density

        

    except:
        pass

print("Block avg", np.mean(means))
plt.plot(range(len(means)), means, label="Diff Avg", linewidth=2)
plt.legend()
plt.ylabel("Density Err (kg/m^3)")
plt.show()

fp = findpeaks(lookahead=1, interpolate=10, verbose=0)
results = fp.fit(means)["df_interp"]

In [ ]:
# file = "density_iters/BA_copy/workspace/1ff6ae9604f00d470071329bfaacbdf0/inter_prod.edr"
# df = panedr.edr_to_df(file)
density = np.loadtxt("example_output_files/sft_calc_test.xvg", comments=["#", "@"])
df = pd.DataFrame(density, columns=["Time", "Pxx", "Pyy", "Pzz", "#Surf*SurfTen"])

def BA_block_avg(sft_gmx_data_df):
    num_blocks_array = np.arange(3,20,1)
    SFT_data_time_length = len(df)
    ave_SFT_value_array = np.zeros([len(num_blocks_array)])
    SFT_var = np.zeros([len(num_blocks_array)])
    SFT_error_array = np.zeros([len(num_blocks_array)])

    for j in range(len(num_blocks_array)):
        num_blocks = num_blocks_array[j]
        sft_values_array = np.zeros([num_blocks])
        for i in range(num_blocks):
            data_length = int(SFT_data_time_length//num_blocks)
            if i == 0:
                sft_val = (sft_gmx_data_df.iloc[-data_length*(i+1): , -1].mean())/20
            else:
                sft_val = (sft_gmx_data_df.iloc[-data_length*(i+1) : -data_length*(i), -1].mean())/20
                
            sft_values_array[i] = sft_val
            
        ave_SFT_value_array[j] = np.mean(sft_values_array)
        SFT_value_var_unormalized = np.var(sft_values_array)
        SFT_value_var_normalized = SFT_value_var_unormalized/(num_blocks - 1)
        SFT_var[j] = SFT_value_var_normalized
        SFT_value_err = np.sqrt(SFT_value_var_normalized)
        SFT_error_array[j] = SFT_value_err
        
    index_max = np.argmax(SFT_error_array)
    final_num_blocks = num_blocks_array[index_max]
    final_SFT_err = SFT_error_array[index_max]
    final_SFT_val = ave_SFT_value_array[index_max]
    final_SFT_var = SFT_var[index_max]

    # with job:
    #     np.savetxt("SFT_final_ave_value_DIR_INDEX.txt", np.array([final_SFT_val]))
    #     np.savetxt("SFT_final_ave_value_error_DIR_INDEX.txt", np.array([final_SFT_err]))
    #     np.savetxt("SFT_final_num_blocks_DIR_INDEX.txt", np.array([final_num_blocks]))

    return ave_SFT_value_array, SFT_var, SFT_error_array

print(df[["Time","#Surf*SurfTen"]].head())
(means_est2, vars_est2, vars_err2) = BA_block_avg(df[["Time","#Surf*SurfTen"]])

print(means_est2[0], max(np.sqrt(vars_est2)))

In [ ]:
sim_name = "inter_eq_cen"
property = "Meandiff"
eq_col_file = sim_name + "_" + property + ".csv"
eq_data_dict = {}
eq_data_dict[property] = {
    "data": results['y'],
    "time_data": results['x'],
    "file": eq_col_file,
}

In [ ]:
def plot_res_pymser(t_col, eq_col, results, name):
    fig, [ax1, ax2] = plt.subplots(
        1, 2, gridspec_kw={"width_ratios": [2, 1]}, sharey=True
    )

    ax1.set_ylabel(name, color="black", fontsize=14, fontweight="bold")
    ax1.set_xlabel("Time (ps)", fontsize=14, fontweight="bold")

    ax1.plot(t_col, eq_col, label="Raw data", color="blue")

    ax1.plot(
        t_col[results["t0"] :],
        results["equilibrated"],
        label="Equilibrated data",
        color="red",
    )

    ax1.plot(
        [0, t_col[-1]],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax1.fill_between(
        t_col,
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )
    print("t_col", t_col[results["t0"]])
    # ax1.set_yticks(np.arange(eq_col.min(), eq_col.max(), eq_col.max()/10))
    ax1.set_xlim(t_col.min(), t_col.max())
    ax1.tick_params(axis="y", labelcolor="black")

    ax1.grid(alpha=0.3)
    ax1.legend()

    ax2.hist(
        eq_col,
        orientation="horizontal",
        bins=30,
        edgecolor="blue",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    bin_red = 10
    ax2.hist(
        results["equilibrated"],
        orientation="horizontal",
        bins=bin_red,
        edgecolor="red",
        lw=1.5,
        facecolor="white",
        zorder=3,
    )

    ymax = int(ax2.get_xlim()[-1])

    ax2.plot(
        [0, ymax],
        [results["average"], results["average"]],
        color="green",
        zorder=4,
        label="Equilibrated average",
    )

    ax2.fill_between(
        range(ymax),
        results["average"] - results["uncertainty"],
        results["average"] + results["uncertainty"],
        color="lightgreen",
        alpha=0.3,
        zorder=4,
    )

    ax2.set_xlim(0, ymax)

    ax2.grid(alpha=0.5, zorder=1)

    fig.set_size_inches(9, 5)
    fig.set_dpi(100)
    fig.tight_layout()
    # save_name = "MSER_eq_vol.png"
    # fig.savefig(job.fn(save_name), dpi=300, facecolor="white")
    # plt.close(fig)

def check_equil_converge(eq_data_dict, prod_tol):
    equil_matrix = []
    res_matrix = []
    prod_begin = 0
    prop_names = list(eq_data_dict.keys())
    num_cols = len(prop_names)

    try:
        # Load data for both boxes
        for key in list(eq_data_dict.keys()):
            eq_col = eq_data_dict[key]["data"]
            print("eq col", eq_col)
            print("len eq col", len(eq_col))
            prod_tol = len(eq_col)/4
            print("prod tol", prod_tol)
            batch_size = max(1, int(len(eq_col) * 0.0005))

            # Try with ADF test enabled, fallback without it if it fails
            try:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=True,
                    uncertainty="uSD",
                    print_results=False,
                )
                adf_test_failed = results["critical_values"]["1%"] <= results["adf"]
            except:
                results = pymser.equilibrate(
                    eq_col,
                    LLM=False,
                    batch_size=batch_size,
                    ADF_test=False,
                    uncertainty="uSD",
                    print_results=False,
                )
                results["adf"], results["critical_values"], adf_test_failed = (
                    None,
                    None,
                    False,
                )

            equilibrium = len(eq_col) - results["t0"] >= prod_tol
            print("len eq_col", len(eq_col),  "res t0", results["t0"])
            print("eq_col[results[t0]]", eq_col[results["t0"]])
            prod_begin = results["t0"]
            print("Results prod",results["t0"])
            equil_matrix.append(equilibrium and not adf_test_failed)
            res_matrix.append(results)

        for i, is_equilibrated in enumerate(equil_matrix):
            key_name = list(eq_data_dict.keys())[i]
            col_vals = eq_data_dict[key_name]["data"]
            t_vals = eq_data_dict[key_name]["time_data"]
            print(t_vals.min(), t_vals.max())
            # plot all

            # if not all(equil_matrix):
            plot_res_pymser(
                t_vals, col_vals, res_matrix[i], prop_names[i % num_cols]
            )

            # Display outcome
            prod_cycles = len(col_vals) - res_matrix[i]["t0"]
            if is_equilibrated:
                # Plot successful equilibration
                statement = f"       > Success! Found {prod_cycles} production cycles."
            else:
                # Plot failed equilibration
                statement = f"       > Equil Failure! "
                if res_matrix[i]["adf"] is None:
                    # Note: ADF test failed to complete
                    statement += f"ADF test failed to complete! "
                elif res_matrix[i]["adf"] > res_matrix[i]["critical_values"]["1%"]:
                    adf, one_pct = (
                        res_matrix[i]["adf"],
                        res_matrix[i]["critical_values"]["1%"],
                    )
                    statement += f"ADF value: {adf}, 99% confidence value: {one_pct}! "
                if len(col_vals) - res_matrix[i]["t0"] < prod_tol:
                    statement += f"Only {prod_cycles} production cycles found."
            print(statement)
            print("All matrix eq", all(equil_matrix))
            return prod_begin
            # with open("Equil_Output.txt", "a") as f:
            #     print(statement, file=f)

    except Exception as e:
        # This will cause an error in the GEMC operation which lets us know that the job failed
        raise Exception("Error processing job ")

    



In [ ]:
# print(df["Time"].values)
prod_begin = check_equil_converge(eq_data_dict, prod_tol = None)
# df = panedr.edr_to_df(file)
# volume = df[property].values
# vol_prod = volume[prod_begin:]
# print(np.mean(vol_prod))

In [ ]:
file = "density_iters/npzzat_rect_constr/workspace/a8d9ab6da5766553901652ebd1275481/npzzat_prod.edr"
file = "density_iters/npt_rect_constr/workspace/1c5738d1d38758c4cb79dcf8de3b172c/npt_prod.edr"
file = "density_iters/npt_check_only/workspace/63c3ac10b55a341bf39d5231b9ec62b0/npt_prod.edr"
file = "density_iters/runs_npzzat_rect/workspace/301d869508a6f0f95a991fff8935e7f2/npzzat_prod.edr"

df = panedr.edr_to_df(file)
property = np.array(df["Density"].values)
#Use block averaging to calculate the variance of each property
(means_est, vars_est, vars_err) = block_average(property)
print(means_est[0], max(np.sqrt(vars_est)))

In [ ]:
# density = np.loadtxt("density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_eq_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/BA_copy_reprod/workspace/1ff6ae9604f00d470071329bfaacbdf0/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/npt_check_only/workspace/53790f5d62edcf52debeda815e9dc8e4/inter_prod_density.xvg", comments=["#", "@"])
# density = np.loadtxt("density_iters/runs_npzzat_rect/workspace/e211d0387e1ce02d02bd11448d6088fe/inter_eq_density.xvg", comments=["#", "@"])
density = np.loadtxt("example_output_files/R125_pure_295K_final_rhoMass_profile_System.xvg", comments=["#", "@"])
density = np.loadtxt("example_output_files/sft_calc_density.xvg", comments=["#", "@"])
prop_vals = calc_mass_dens(density)

print(np.mean(prop_vals))

In [ ]:
import re
import numpy as np

# Load the full text from file
with open('density_iters/runs_npzzat_rect/workspace/1f489249dd7f23f12833e7230f16fa4f/inter_prod_ift_st.txt') as f:
    text = f.read()

# Match lines of the form:
# EnergyName   Average  Err.Est.  RMSD  Tot-Drift
# We assume columns are space-separated, first column is the name
data_lines = re.findall(r'-{5,}[\r\n]+([^\r\n]+)', text)

# Parse each line into components
entries = [line.split() for line in data_lines if len(line.split()) >= 5]

# Extract numeric columns
averages = np.array([float(e[1]) for e in entries])
errors   = np.array([float(e[2]) for e in entries])
rmsds    = np.array([float(e[3]) for e in entries])

print("Averages:", averages)
print("Errors:  ", errors)
print("RMSDs:   ", rmsds)

In [ ]:
property = "Density"
df = panedr.edr_to_df("density_iters/runs_npzzat_rect/workspace/301d869508a6f0f95a991fff8935e7f2/npzzat_prod.edr")
print(df.columns)
# df.columns
density = df[property].values
dens_eq = np.mean(density)
print(dens_eq)